# Generation

This notebook uses the collection stored in Milvus by the Task 2 notebook, you must run that notebook before this one.

The final step in a Retrieval-Augmented Generation (RAG) pipeline typically involves generating the final response or output based on the retrieved information and the query. Here’s a breakdown of the entire process and where the final step fits in:

Query Processing: A user's query is first processed, often by an encoder or a model that can understand the context and intent behind the query.

Document Retrieval: Based on the query, the system retrieves relevant documents, passages, or pieces of information from a knowledge base or corpus. This is usually done using a retrieval model (e.g., a dense retriever like Dense Retriever or BM25 for traditional search).

Contextualization (Prompt Engineering): The retrieved documents or passages are then combined with the original query, and potentially encoded into a new context. This step ensures that the relevant information is directly aligned with the query to help generate a more informed response.

Response Generation (Final Step): The final step is the generation phase, where a generative model (e.g., a transformer model like Llama or GPT) takes the combined query and retrieved context to produce a coherent and contextually relevant response. This step utilizes both the original query and the additional context from the retrieved documents to craft a response that answers the query effectively.

In short, the final step in a RAG pipeline is generating the final output or answer, which leverages both the query and the retrieved knowledge to create an informative, accurate, and contextually relevant response. This final generation is typically handled by a language model (e.g. `meta/llama-3.3-70b-instruct`)  that synthesizes information from the retrieval stage and ensures that the output aligns well with the user's intent.

<img src="https://blogs.nvidia.com/wp-content/uploads/2024/11/ragexplainer131-960x1143.png" width="600" />


We need to identify a question we would like to ask. To generate a meaningful answer the query must have an answer in the corpus. Otherwise, you may find yourself getting an answer that was hallucinated, or incorrect. This is why loading the corpus of data you want to ask questions about is so important. You need to make sure that there are answers in your corpus for the types of questions you want to ask. In this case we are working with the BEIR - Natural Questions dataset. This dataset was created by google, it consists of wikipedia information and related to real questions asked by users. Some of the questions asked are:  
- "when are hops added to the brewing process?" 
- "where is the world s largest ice sheet located today?" 
- "where is blood pumped after it leaves the right ventricle?" 
- "who is the voice of tony the tiger?" 
- "where does the energy in a nuclear explosion come from?"

In [1]:
query = "what color is the sky?"
query = "where is blood pumped after it leaves the right ventricle?"

In [2]:
from openai import OpenAI

api_key = "$API_KEY_REQUIRED_IF_EXECUTING_OUTSIDE_NGC"
openai_client = OpenAI(
  base_url = "http://embedding:8000/v1",
  api_key = api_key
)

In [3]:
import pymilvus
embeddings = openai_client.embeddings.create(
      input=[query],
      model="nvidia/llama-3.2-nv-embedqa-1b-v2",
      encoding_format="float",
      extra_body={"input_type": "query", "truncate": "END"}  
)
embeddings = [entry.embedding for entry in embeddings.data]
embeddings[0][:10], len(embeddings[0])

([-0.0101470947265625,
  -0.009033203125,
  0.003429412841796875,
  -0.033843994140625,
  0.02618408203125,
  -0.040283203125,
  -0.043975830078125,
  -0.01593017578125,
  0.00603485107421875,
  0.023193359375],
 2048)

In [4]:
client = pymilvus.MilvusClient(uri="http://milvus:19530")
result_ids = client.search(
    "beir_nq",
    embeddings,
    search_params={"ef": 40},
    anns_field="embedding",
    limit=10 # top_k
)

In [5]:
results_ids = [res["id"] for res in result_ids[0]]
results_ids

[64307, 35517, 64310, 64308, 64311, 921992, 971685, 35496, 921989, 220613]

In [6]:
import pickle 
records = pickle.load(open("data/records.pickle", "rb"))
query_texts = [records[res_id]["text"] for res_id in results_ids]
query_texts

['Deoxygenated blood leaves the heart, goes to the lungs, and then re-enters the heart; Deoxygenated blood leaves through the right ventricle through the pulmonary artery. From the right atrium, the blood is pumped through the tricuspid valve (or right atrioventricular valve), into the right ventricle. Blood is then pumped from the right ventricle through the pulmonary valve and into the main pulmonary artery.',
 'The right heart collects deoxygenated blood from two large veins, the superior and inferior venae cavae. Blood collects in the right and left atrium continuously.[7] The superior vena cava drains blood from above the diaphragm and empties into the upper back part of the right atrium. The inferior vena cava drains the blood from below the diaphragm and empties into the back part of the atrium below the opening for the superior vena cava. Immediately above and to the middle of the opening of the inferior vena cava is the opening of the thin-walled coronary sinus.[7] Additionall

In [7]:
import os

def get_answer(
    query,
    chunk_answers,
    endpoint:str = "https://integrate.api.nvidia.com/v1", 
    model_name: str = "meta/llama-3.3-70b-instruct",
    api_key: str = None, 
):
    api_key = api_key or os.environ.get("NVIDIA_API_KEY", None)
    api_key = "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"
    client = OpenAI(
      base_url = endpoint,
      api_key = api_key
    )
    
    completion = client.chat.completions.create(
      model=model_name,
      messages=[{"role":"user","content":f"Answer the following question. {query} With the following information: {chunk_answers}"}],
      temperature=0.2,
      top_p=0.7,
      max_tokens=1024,
      stream=True
    )
    
    for chunk in completion:
      if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")


In [8]:
get_answer(
    query,
    query_texts,
)

After it leaves the right ventricle, blood is pumped into the pulmonary artery, which then divides into the left and right main pulmonary arteries, one for each lung. These arteries branch into smaller pulmonary arteries that spread throughout the lungs, where gas exchange occurs, allowing oxygen to be absorbed into the blood and carbon dioxide to be released.

In [14]:
query_text="what color is the sky?"
get_answer(
    "what game ships with the nintendo switch 2?",
    "Mario Kart World ships with the nintendo switch 2",
)

According to the information provided, the game that ships with the Nintendo Switch 2 is Mario Kart World.